In [1]:
import os
import torch
import pandas as pd

In [6]:
base_dir = "/gws/nopw/j04/wiser_ewsa/mrakotomanga/OB/ncast/checkpoints"

rows = []

for folder in os.listdir(base_dir):

    if not folder.startswith("t1_lr"):
        continue

    ckpt_path = os.path.join(base_dir, folder, "best-ncast.ckpt")

    if not os.path.exists(ckpt_path):
        continue

    ckpt = torch.load(ckpt_path, map_location="cpu")

    callbacks = ckpt["callbacks"]

    for key in callbacks:

        if "ModelCheckpoint" in key:

            val_auc = callbacks[key]["best_model_score"].item()

            rows.append({
                "run": folder,
                "val_auc": val_auc
            })

            break

df = pd.DataFrame(rows)

df = df.sort_values("val_auc", ascending=False)

print(df.to_string(index=False))

                           run  val_auc
t1_lr0.0002_do0.2_pw25.0_a0.3_ 0.976286
t1_lr0.0001_do0.1_pw25.0_a0.3_ 0.974890
t1_lr0.0001_do0.2_pw25.0_a0.1_ 0.974496
t1_lr0.0001_do0.2_pw10.0_a0.3_ 0.973945
t1_lr0.0001_do0.2_pw25.0_a0.3_ 0.973209
t1_lr0.0001_do0.2_pw25.0_a0.5_ 0.972884
t1_lr0.0001_do0.2_pw50.0_a0.3_ 0.972719
 t1_lr5e-05_do0.2_pw25.0_a0.3_ 0.971769


In [ ]:
auc_csv = "/home/users/mendrika/NCAST/Output/optimisation/ncast/optimisation-t1-auc.csv"
fss_csv = "/home/users/mendrika/NCAST/Output/optimisation/ncast/optimsation-t1-fss.csv"

auc_df = pd.read_csv(auc_csv)
fss_df = pd.read_csv(fss_csv)

rows = []

for auc_col in auc_df.columns:

    if " - val_auc" not in auc_col:
        continue

    if "__MIN" in auc_col or "__MAX" in auc_col:
        continue

    run = auc_col.split(" - ")[0].strip()

    if not run.startswith("t1_lr"):
        continue

    fss_col = f"{run} - val_fss_9"

    if fss_col not in fss_df.columns:
        print(f"Missing FSS column for {run}")
        continue

    best_idx = auc_df[auc_col].idxmax()

    rows.append({
        "run": run,
        "val_auc": auc_df.loc[best_idx, auc_col],
        "val_fss": fss_df.loc[best_idx, fss_col]
    })

summary = pd.DataFrame(rows)

summary = summary.sort_values(
    "val_auc",
    ascending=False
)

print(summary.to_string(index=False))

                           run  val_auc  val_fss
t1_lr0.0002_do0.2_pw25.0_a0.3_ 0.976286 0.253628
t1_lr0.0001_do0.1_pw25.0_a0.3_ 0.974855 0.246183
t1_lr0.0001_do0.2_pw25.0_a0.1_ 0.974496 0.263372
t1_lr0.0001_do0.2_pw10.0_a0.3_ 0.973945 0.268473
t1_lr0.0001_do0.2_pw25.0_a0.3_ 0.973209 0.239833
t1_lr0.0001_do0.2_pw25.0_a0.5_ 0.972884 0.230556
t1_lr0.0001_do0.2_pw50.0_a0.3_ 0.972719 0.195632
 t1_lr5e-05_do0.2_pw25.0_a0.3_ 0.971769 0.243506
